# 🧠 Quillan-Ronin v5.4.0 ONI — 500M Model Training Notebook

Optimized for **Google Colab (Tesla T4, A100, or L4)** with automatic Google Drive checkpoint synchronization, PyTorch Automatic Mixed Precision (AMP), gradient checkpointing, and automatic session resume.

## 1. Verify GPU Acceleration & Tensor Core Capability

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU: {gpu_name} | VRAM: {vram_gb:.2f} GB")

## 2. Mount Google Drive for Persistent Checkpoints
Colab sessions can timeout or disconnect. Saving checkpoints directly to Google Drive guarantees zero lost compute.

In [ ]:
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')
DRIVE_CKPT_DIR = Path('/content/drive/MyDrive/Quillan_Checkpoints')
DRIVE_LOG_DIR = Path('/content/drive/MyDrive/Quillan_Logs')
DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_LOG_DIR.mkdir(parents=True, exist_ok=True)
print(f"Persistent checkpoint directory ready: {DRIVE_CKPT_DIR}")

## 3. Clone Repository & Install Dependencies

In [ ]:
%cd /content
!git clone https://github.com/sneed-and-feed/Quillan-Ronin.git
%cd /content/Quillan-Ronin
!pip install -q tokenizers numpy einops psutil scipy

## 4. Prepare Dataset
Convert your raw `.txt` or `.jsonl` files into memory-mapped binary arrays (`train_ids.bin`, `val_ids.bin`).
*(If no raw dataset is uploaded yet, you can pass `--synthetic-data` to benchmark hardware throughput)*

In [ ]:
# Example: Pack dataset from Google Drive or local upload
# !python oni/prepare_data.py --input /content/drive/MyDrive/QuillanData/my_corpus.jsonl --output-dir /content/data

# Alternatively, create a demo dataset to verify the pipeline:
import numpy as np
from pathlib import Path
demo_dir = Path('/content/data')
demo_dir.mkdir(parents=True, exist_ok=True)
if not (demo_dir / 'train_ids.bin').exists():
    print("Creating initial validation dataset tokens...")
    sample_tokens = np.random.randint(0, 50257, size=1000000, dtype=np.uint16)
    sample_labels = np.roll(sample_tokens, -1).astype(np.int32)
    sample_tokens[:950000].tofile(demo_dir / 'train_ids.bin')
    sample_labels[:950000].tofile(demo_dir / 'train_labels.bin')
    sample_tokens[950000:].tofile(demo_dir / 'val_ids.bin')
    sample_labels[950000:].tofile(demo_dir / 'val_labels.bin')
    print(f"Demo dataset created in {demo_dir} (1,000,000 tokens)")

## 5. Launch 500M Model Training

- **Architecture**: `n_layer=18` yields ~492M parameters
- **Precision**: `--amp` enables PyTorch Automatic Mixed Precision with `GradScaler`
- **Memory**: `--grad-checkpoint` preserves VRAM
- **Checkpoints**: `--ckpt-dir` writes directly to Google Drive
- **Resume**: `--resume` seamlessly reloads model and optimizer state if interrupted

In [ ]:
!python oni/train_oni.py \
    --n-layer 18 \
    --seq-len 512 \
    --batch-size 2 \
    --grad-accum 8 \
    --lr 3e-4 \
    --warmup 200 \
    --steps 20000 \
    --eval-every 250 \
    --sample-every 500 \
    --save-every 250 \
    --data-dir /content/data \
    --ckpt-dir /content/drive/MyDrive/Quillan_Checkpoints \
    --log-dir /content/drive/MyDrive/Quillan_Logs \
    --amp \
    --grad-checkpoint \
    --device cuda \
    --resume

## 6. Test Model Inference & Sample Generation

In [ ]:
import sys
from pathlib import Path
import torch
sys.path.insert(0, '/content/Quillan-Ronin/oni')
from quillan_tokenizer_unified import UnifiedQuillanTokenizer
from quillan_v5_4_oni import QuillanOniConfig, QuillanRoninOni

tok = UnifiedQuillanTokenizer()
cfg = QuillanOniConfig(n_layer=18, max_seq_len=512)
model = QuillanRoninOni(cfg).cuda()

ckpt_path = Path('/content/drive/MyDrive/Quillan_Checkpoints/quillan_oni_latest.pt')
if ckpt_path.exists():
    print(f"Loading checkpoint: {ckpt_path}")
    ck = torch.load(ckpt_path, map_location='cuda')
    model.load_state_dict(ck['model'], strict=False)
    print(f"Loaded model from step {ck.get('step', '?')}, val_loss {ck.get('best_val', '?'):.4f}")

prompt = "User: What is the core philosophy of the Quillan-Ronin Council?\n\nAssistant:"
input_ids = tok.encode(prompt, domain='dialogue')
output_ids = model.generate(input_ids, max_tokens=100, temp=0.7)
print("\n--- Generated Response ---\n")
print(tok.decode(output_ids))